# FS-005 Phase 5 — full-cohort run (Colab / Kaggle T4)

Runs the real MedGemma-4B and Qwen2.5-VL-3B models through the region-grounded, uncertainty-aware pipeline in this order:

1. **Validation FIT + ASSESSMENT** generations (N=5, T=0.7) -> **calibrate** weights and tau (fit split only)
2. **Test** generations (N=5) and the Condition-1 greedy run (Track C)
3. **Evaluate** with the frozen weights, Conditions 1-3, Tracks B and C

Everything is **resumable**: artifacts are written to Google Drive after every question, so after a disconnect just re-run the same cell. The test split is never read before `calibrate` has written `frozen.json`.

**Before you start**
- Runtime -> Change runtime type -> **T4 GPU**.
- Add a Colab Secret named `HF_TOKEN` (read access) and accept the MedGemma licence at https://huggingface.co/google/medgemma-4b-it.
- Expect many hours: 5 generations x (719 + 172 validation + up to 945 test) questions x 2 models x 2 tracks. Run one model/track per session if needed; `status` shows progress.

In [ ]:
# 1. Drive + repo + dependencies (does not touch Colab's torch/CUDA)
from google.colab import drive, userdata
import os, subprocess, sys
drive.mount('/content/drive')
OUT = '/content/drive/MyDrive/MedicalVQA/fs005_phase5_full'
os.makedirs(OUT, exist_ok=True)

if not os.path.exists('/content/MedicalVQA'):
    subprocess.run(['git','clone','https://github.com/Abishakm1507/MedicalVQA.git','/content/MedicalVQA'], check=True)
%cd /content/MedicalVQA
subprocess.run([sys.executable,'-m','pip','install','-q','pydantic>=2.6','pillow','numpy<2','transformers>=4.46','accelerate','bitsandbytes','safetensors','huggingface_hub','qwen-vl-utils'], check=True)

from huggingface_hub import login
login(token=userdata.get('HF_TOKEN'), add_to_git_credential=False)   # token is never printed
import torch; print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# 2. Safety check: the code runs and the data is where it should be
!python -m pytest tests/unit/test_agreement_uses_parsed_answers.py tests/unit/test_full_run_driver.py -q -p no:cacheprovider 2>&1 | tail -3
!python scripts/fs005_phase5_full_run.py status --out-dir {OUT}

## Step A — validation generations and calibration
Change `MODEL` and `TRACK` and re-run this section for each of: (medgemma-4b, B), (medgemma-4b, C), (qwen2.5-vl-3b, B), (qwen2.5-vl-3b, C).
Use `LIMIT = 20` first as a smoke test on the real model, then set it back to `0` for the full cohort.

In [ ]:
MODEL = 'medgemma-4b'      # or 'qwen2.5-vl-3b'
TRACK = 'B'                # or 'C'
LIMIT = 20                 # 0 = full cohort

In [ ]:
!python scripts/fs005_phase5_full_run.py generate --model {MODEL} --track {TRACK} --split val_fit --limit {LIMIT} --out-dir {OUT}
!python scripts/fs005_phase5_full_run.py generate --model {MODEL} --track {TRACK} --split val_assess --limit {LIMIT} --out-dir {OUT}

In [ ]:
# Fits w1,w2,w3 and tau on val_fit ONLY, reports on val_assess, writes frozen.json
!python scripts/fs005_phase5_full_run.py calibrate --model {MODEL} --track {TRACK} --out-dir {OUT}

## Step B — test generations and evaluation
Only run after `frozen.json` exists for this model/track. Track C also needs the Condition-1 greedy run.

In [ ]:
!python scripts/fs005_phase5_full_run.py generate --model {MODEL} --track {TRACK} --split test --limit {LIMIT} --out-dir {OUT}
if TRACK == 'C':
    !python scripts/fs005_phase5_full_run.py generate --model {MODEL} --track C --split test_c1 --limit {LIMIT} --out-dir {OUT}

In [ ]:
!python scripts/fs005_phase5_full_run.py evaluate --model {MODEL} --track {TRACK} --out-dir {OUT}
!python scripts/fs005_phase5_full_run.py status --out-dir {OUT}

## Step C — hand the results back
Results are under `MyDrive/MedicalVQA/fs005_phase5_full/evaluation/<model>/<track>/condition{1,2,3}/` (`experiment_metrics.json`, per-sample CSV, accuracy-vs-coverage CSV, report). Download that folder and give it to Claude Code to rebuild the paper tables and figures.

In [ ]:
!cd {OUT} && zip -qr /content/fs005_results.zip evaluation calibration && ls -la /content/fs005_results.zip
from google.colab import files
files.download('/content/fs005_results.zip')